# ReWear AI — EDA final en 10 minutos

**Objetivo:** entender qué datos tenemos, qué problemas encontramos y qué vamos a entrenar después. Este notebook integra el trabajo de los cuatro integrantes; los cálculos extensos quedan en los reportes enlazados.

> En cada sección seguimos: **pregunta → evidencia → decisión**. Todavía no se entrenó ningún modelo.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
required = [
    ROOT / "reports/cisutac/eda_cisutac.md",
    ROOT / "reports/raw_fabrid/eda_raw_fabrid.md",
    ROOT / "reports/fabrid/eda_fabrid.md",
    ROOT / "reports/protocolo_experimental.md",
]
missing = [str(path.relative_to(ROOT)) for path in required if not path.exists()]
assert not missing, f"Faltan resultados del EDA: {missing}"
print("EDA integrado listo: 4 bloques y protocolo encontrados.")

## Cinco conceptos, en fácil

- **Caja (bounding box):** rectángulo alrededor del defecto; una **máscara** marca exactamente sus píxeles.
- **Normal:** en detección de anomalías significa una tela sin defecto verificado.
- **PatchCore:** aprende el aspecto de telas normales y resalta las zonas que se ven distintas.
- **Tile:** recorte de una foto grande que conserva más detalle que achicar toda la imagen.
- **Cambio de dominio:** el modelo se entrena con un tipo de foto y se prueba con otro, así que puede fallar aunque el defecto se llame igual.

**Mapa rápido de datos**

| Dataset | Qué cuenta como dato independiente | Etiqueta útil | Uso decidido |
|---|---:|---|---|
| CISUTAC Holes / Spots | 407 / 486 fotos originales | Cajas | Detector en prendas usadas |
| RAW-FABRID | 709 capturas; 19.852 patches relacionados | Máscaras | PatchCore principal |
| FabriD v3 | 2.407 fotos RGB | Polígonos + clase | Segmentación y comparación común |
| FabricSpotDefect | 1.014 originales | Cajas de `spot` | Complemento de manchas |
| RISE | 3 muestras locales | Etiqueta débil, sin localización | No usar por ahora |

## Integrante 1 — CISUTAC: ¿el resize borra los defectos?

**Evidencia.** Al contar sólo originales hay **407 fotos / 785 agujeros** y **486 fotos / 2.498 manchas**. La mediana del lado equivalente es 24,7 px para agujeros y 18,2 px para manchas; al llevar la foto completa a 640 px baja a **4,2 px y 3,0 px**. El 71 % de agujeros y 77 % de manchas quedan por debajo de 8 px.

> **Lado equivalente:** lado de un cuadrado con la misma área que la caja; resume el tamaño del defecto en un solo número.

![Efecto del resize sobre defectos pequeños](../reports/cisutac/cis4_zoom_resize.png)

**Decisión.** Entrenar Holes y Spots por separado, con originales, y comparar foto completa a 640 contra tiles. CISUTAC no tiene prendas normales verificadas, por lo que no sirve para entrenar PatchCore. [Detalle CISUTAC](../reports/cisutac/eda_cisutac.md).

## Integrante 2 — RAW-FABRID: ¿sirve para PatchCore?

**Evidencia.** Hay **507 originales normales y 202 defectuosos**. PatchCore puede entrenar con 14.196 patches `train/good` y evaluarse con 4.969 `test/good` + 687 `test/defect`. Los patches de test se agrupan en sólo 202 capturas madre: no son 5.656 observaciones independientes. Dos máscaras extra no tienen imagen ni metadatos asociados.

![Relación entre patches y captura madre](../reports/raw_fabrid/raw2_parches.png)

> **Line-scan:** cámara industrial que registra la tela línea por línea mientras avanza, con condiciones mucho más controladas que una foto de ropa.

**Decisión.** Usarlo como benchmark principal de PatchCore; calcular incertidumbre agrupando por captura madre y mostrar resultados por rollo. Preservar pero excluir las dos máscaras huérfanas. [Detalle RAW-FABRID](../reports/raw_fabrid/eda_raw_fabrid.md).

## Integrante 3 — FabriD y FabricSpotDefect: ¿permiten una comparación justa?

**Evidencia FabriD.** Tiene **2.407 imágenes**, cuatro defectos y 620 normales declaradas. Se confirmaron 41 derivados de 14 capturas repartidos entre splits; el manifiesto agrupado evita esa fuga. Quedan 607 normales concordantes y se excluyen 19 casos ambiguos. Además, 372/467 ejemplos de marca de aguja son 512×512 y no hay normales con esa resolución: el modelo podría aprender el lote en vez del defecto.

> **Marca de aguja:** pequeñas perforaciones repetidas; **hilo faltante:** ausencia de un hilo que deja una línea o hueco en el tejido.

![Tamaños por clase en FabriD](../reports/fabrid/fab3_tamanos_por_clase.png)

**Evidencia FabricSpotDefect.** Sus 3.288 anotaciones son manchas: la clase `Hole-EnBF` está declarada, pero tiene cero instancias. Las 2.300 imágenes aumentadas en COCO y YOLO son los mismos ejemplos en dos formatos.

**Decisión.** Comparar segmentador y PatchCore sólo en el mismo test FabriD, usando mapas binarios compatibles. FabricSpotDefect queda como apoyo para manchas, no como núcleo. [Detalle FabriD/FSD](../reports/fabrid/eda_fabrid.md).

## Integrante 4 — Integración: los dominios no son intercambiables

**Pregunta.** ¿Podemos entrenar en tela industrial y usar el modelo directamente sobre prendas?

![Comparación visual de dominios](../reports/integracion/comparacion_dominios.png)

| Transferencia sin ajuste | Decisión | Motivo |
|---|---|---|
| FabriD `hole` → CISUTAC Holes | Viable con limitaciones | Clase compatible; cambia mucho la escala y la captura |
| FabriD `stain` → CISUTAC Spots | Viable con limitaciones | Mapeo aproximado; convertir regiones a cajas |
| FabricSpotDefect → CISUTAC Spots | Viable con limitaciones | Ambas usan cajas de manchas, pero el encuadre cambia |
| RAW-FABRID → CISUTAC | Sólo exploratorio | CISUTAC no tiene ropa sana para medir falsos positivos |

**Decisión.** No mezclar datasets. Para medir transferencia se mantiene una misma tarea y métrica; si las salidas no son compatibles, mostrar sólo ejemplos cualitativos. [Tabla completa](../reports/integracion/factibilidad_transferencia.csv).

## Integridad: cómo evitamos resultados engañosos

> **Fuga de datos:** una foto o versión casi igual aparece en train y test, por lo que el modelo parece generalizar aunque ya vio ese contenido.

| Dataset | Regla congelada |
|---|---|
| CISUTAC | Originales solamente; split agrupado por día/sesión: Holes 300/43/64 y Spots 361/59/66 |
| RAW-FABRID | Separación por `parent_raw`; medir incertidumbre por madre y desglosar por rollo |
| FabriD | Manifiesto agrupado por `source_id`; excluir 19 ambiguos |
| FabricSpotDefect | Aumentaciones sólo en train; fotos sin cajas no se suponen sanas |

El test no se usa para elegir tiles, umbrales ni hiperparámetros. Si se descubre una relación nueva entre capturas, se versiona el manifiesto **antes** de entrenar y nunca mirando el resultado final.

## Qué hacemos después del EDA

1. **Supervisado:** YOLO11n separado para Holes y Spots. Comparar imagen completa 640 vs. tiles 640 con 20 % de solapamiento inicial. Medir mAP, precision, recall y resultados por tamaño.
2. **Anomalías:** PatchCore en RAW-FABRID. Medir AUROC por patch, AUROC por píxel y AUPRO, agrupando por captura madre.
3. **Comparación justa:** segmentador vs. PatchCore en el mismo test FabriD y con mapas binarios.
4. **Opcional:** transferencia FabriD → CISUTAC sin ajuste, después de los tres experimentos centrales.

> **AUPRO:** mide cuánto cubre el mapa las zonas defectuosas sin pintar demasiado tejido sano. `mAP` y `AUROC` miden cosas distintas y no deben compararse como una única tabla de posiciones.

**Hipótesis principal:** los tiles deberían recuperar defectos pequeños que se vuelven casi invisibles al achicar la prenda, aunque costarán más tiempo de procesamiento. El EDA justifica probarlo; todavía no demuestra que funcione.

## Cierre en cuatro líneas

- **Dataset más cercano al problema real:** CISUTAC.
- **Mejor dataset para PatchCore:** RAW-FABRID.
- **Lugar para comparar supervisado vs. anomalías:** FabriD, con el mismo test.
- **Aporte propio medible:** preservar detalle con tiles y medir mejora en defectos pequeños contra su costo.

Antes de entrenar, los cuatro integrantes deben revisar y aprobar el [protocolo experimental](../reports/protocolo_experimental.md). El resumen para el informe está en [conclusiones del EDA](../reports/eda_conclusiones.md).